In [ ]:

# %pip install langgraph langchain-core langchain-openai
# %pip install rich python-dotenv
# %pip install tavily-python langchain-tavily
# %pip install langgraph-checkpoint-sqlite aiosqlite

### Defining state

 - Think of this as the "contract" for your entire pipeline.
 - Every node can read ANY field, and return updates to ANY field.

 - Why TypedDict and not a regular dict?
    - It gives you type hints, autocomplete, and catches typos.
    - LangGraph uses it to understand what your state looks like.

In [ ]:
from typing import TypedDict, Optional, Any

class ContentState(TypedDict):
    # ── Core content fields ──────────────────────────────
    topic:           str            # The topic to research and write about
    research_notes:  str            # Filled by Researcher
    draft:           str            # Filled by Writer (updated each revision)
    review_score:    Optional[int]  # Filled by Reviewer
    feedback:        str            # Filled by Reviewer
    
    # ── New fields for revision tracking ─────────────────
    revision_count:   int           # How many times Writer has been called
    review_data:      Any           # Full JSON review breakdown (dict)
    revision_history: list          # Log of [{revision, score, feedback}]

In [ ]:
import json
import re
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from rich import print as rprint

load_dotenv()  

api_key = os.getenv("OPEN_ROUTER_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")


llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free", 
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost:3000", # Required by OpenRouter for some models
        "X-Title": "LangGraph App",
    }
)

### Researcher

In [ ]:
def researcher_node(state: ContentState) -> dict:
    """
    The Researcher node:
    - Reads:  'topic' from state
    - Writes: 'research_notes' to state
    
    How it works:
    1. LLM with tools decides WHAT to search
    2. We execute the search
    3. LLM synthesizes results into clean research notes
    
    This is the "ReAct" pattern:
    Reason → Act (use tool) → Observe results → Reason again
    """
    print("\n🔍 RESEARCHER NODE ACTIVATED")
    
    
    topic = state["topic"]
    
    # ── Step A: Set up LLM with Tool ─────────────────────────
    search_tool = TavilySearch(max_results=3)
    llm_with_tools = llm.bind_tools([search_tool])
    
    # ── Step B: Ask the LLM what to search for ───────────────
    print(f"📌 Topic to research: {topic}")
    
    messages = [
        SystemMessage(content="""You are a research assistant. 
Your job is to search for information about the given topic.
Use the search tool to find relevant, up-to-date information."""),
        
        HumanMessage(content=f"""Research this topic thoroughly: {topic}
        
Please search for:
1. Key concepts and definitions
2. Current trends and developments  
3. Important facts and statistics
4. Expert opinions or analysis

Use the search tool now."""),
    ]
    
    # First LLM call: LLM decides to call the tool
    print("🤖 LLM deciding what to search for...")
    llm_response = llm_with_tools.invoke(messages)
    
    print(f"🔧 Tool calls requested: {len(llm_response.tool_calls)}")
    
    # ── Step C: Execute the Tool Calls ───────────────────────
    # The LLM told us WHAT to search — now we actually DO the search
    
    all_search_results = []  # Collect all raw results here
    tool_messages = []       # These go back to the LLM for synthesis
    
    for tool_call in llm_response.tool_calls:
        print(f"🌐 Executing search: '{tool_call['args'].get('query', 'N/A')}'")
        
        # Actually run the Tavily search
        search_results = search_tool.invoke(tool_call["args"])
        
        # search_results is a list of dicts with keys:
        # 'url', 'content', 'title', 'score'
        all_search_results.append(search_results)
        
        print(f"   ✅ Got {len(search_results)} results")
        
        # Wrap results in a ToolMessage so the LLM can read them
        # The tool_call_id links this result back to the specific tool call
        tool_messages.append(
            ToolMessage(
                content=str(search_results),
                tool_call_id=tool_call["id"],
            )
        )
    
    # ── Step D: LLM Synthesizes the Results ──────────────────
    # Now feed the raw search results BACK to the LLM
    # Ask it to write clean, structured research notes
    
    print("📝 LLM synthesizing search results into research notes...")
    
    # Build the full conversation: original messages + LLM response + tool results
    synthesis_messages = tool_messages + [
        HumanMessage(content=f"""Based on these search results, write comprehensive 
research notes about: {topic}

Structure your notes as:
## Key Findings
[main points discovered]

## Important Facts & Data  
[specific statistics, dates, numbers]

## Current Trends
[what's happening now]

## Summary
[2-3 sentence overview]

Be specific and cite the sources where relevant.""")
    ]
    
    # Second LLM call: pure synthesis, no more tool calls needed
    final_response = llm.invoke(synthesis_messages)
    
    research_notes = final_response.content
    
    print(f"✅ Research complete! Notes: {len(research_notes)} characters")
    print("─" * 50)
    
    return {"research_notes": research_notes}

### The writer

In [ ]:


def writer_node(state: ContentState) -> dict:
    """
    UPDATED Writer node — now fully revision-aware.
    
    - Reads:  'topic', 'research_notes', 'feedback', 
              'draft' (previous), 'revision_count', 'revision_history'
    - Writes: 'draft', 'revision_count', 'revision_history'
    
    Two modes:
      Mode A (First draft):  revision_count == 0, no feedback yet
      Mode B (Revision):     revision_count > 0, has specific feedback to address
    """
    print("\n✍️  WRITER NODE ACTIVATED")
    
    
    topic            = state["topic"]
    research_notes   = state["research_notes"]
    feedback         = state.get("feedback", "")
    previous_draft   = state.get("draft", "")
    revision_count   = state.get("revision_count", 0)
    revision_history = state.get("revision_history", [])
    review_data      = state.get("review_data", {})
    
    print(f"📝 Draft attempt #{revision_count + 1}")
    
    # ── Choose the right prompt based on mode ────────────────
    if revision_count == 0:
        # ── MODE A: FIRST DRAFT ───────────────────────────────
        print("✨ Mode: First draft from research notes")
        
        prompt = f"""You are a professional content writer at a top publication.

TOPIC: {topic}

RESEARCH NOTES (your ONLY source of facts — use them!):
{research_notes}

YOUR TASK:
Write an exceptional article based on the research notes above.

REQUIREMENTS:
- Hook the reader in the first sentence
- Use specific facts, data points, and examples from the research notes
- Structure: Strong intro → 2-3 substantive body paragraphs → Memorable conclusion
- Write for an intelligent general audience (no unnecessary jargon)
- Be specific, not generic — avoid clichés like "In today's fast-paced world..."
- Length: 400-600 words

Write the article now:"""

    else:
        # ── MODE B: REVISION ──────────────────────────────────
        print(f"🔄 Mode: Revision #{revision_count} — addressing editor feedback")
        
        # Build a history summary so the writer knows what's been tried
        history_summary = ""
        if revision_history:
            history_summary = "\n\nPREVIOUS REVISION SCORES:\n"
            for h in revision_history:
                history_summary += f"  Revision {h['revision']}: {h['score']}/10\n"
        
        # Extract specific category scores if available
        score_breakdown = ""
        if review_data and "scores" in review_data:
            scores = review_data["scores"]
            score_breakdown = f"""
SPECIFIC SCORE BREAKDOWN (focus on the low scores!):
  Accuracy:     {scores.get('accuracy', '?')}/2
  Clarity:      {scores.get('clarity', '?')}/2  
  Structure:    {scores.get('structure', '?')}/2
  Engagement:   {scores.get('engagement', '?')}/2
  Completeness: {scores.get('completeness', '?')}/2"""
        
        strengths_text = ""
        if review_data and "strengths" in review_data:
            strengths_text = "\nSTRENGTHS TO KEEP:\n" + "\n".join(
                f"  ✓ {s}" for s in review_data["strengths"]
            )
        
        issues_text = ""
        if review_data and "critical_issues" in review_data:
            issues_text = "\nCRITICAL ISSUES TO FIX:\n" + "\n".join(
                f"  ✗ {i}" for i in review_data["critical_issues"]
            )
        
        prompt = f"""You are a professional writer doing a REVISION.

TOPIC: {topic}

RESEARCH NOTES (your source of facts):
{research_notes}

{'─'*40}
YOUR PREVIOUS DRAFT (revision #{revision_count - 1}):
{previous_draft}

{'─'*40}
EDITOR'S FEEDBACK TO ADDRESS:
{feedback}
{score_breakdown}
{strengths_text}
{issues_text}
{history_summary}
{'─'*40}

YOUR TASK:
Rewrite the article to address ALL of the editor's feedback.

IMPORTANT RULES:
1. Keep what the editor praised (the strengths listed above)
2. Fix EVERY critical issue mentioned
3. Use more specific facts from the research notes
4. Do NOT just make small tweaks — if the editor wants big changes, make them
5. Aim for the same length (400-600 words) but make it significantly better
6. Do NOT mention this is a revision

Write the improved article now:"""

    print("🤖 LLM generating the draft...")
    response = llm.invoke(prompt)
    new_draft = response.content
    
    # ── Update revision history ───────────────────────────────
    # Log the PREVIOUS draft's results before overwriting
    if revision_count > 0 and previous_draft:
        revision_history = revision_history + [{
            "revision": revision_count,
            "score":    state.get("review_score", "N/A"),
            "feedback": feedback[:150] + "..." if len(feedback) > 150 else feedback,
        }]
    
    print(f"✅ Draft #{revision_count + 1} complete ({len(new_draft)} characters)")
    print("─" * 50)
    
    return {
        "draft":            new_draft,
        "revision_count":   revision_count + 1,
        "revision_history": revision_history,
    }

### Reviewer node

In [ ]:
def reviewer_node(state: ContentState) -> dict:
    """
    UPGRADED Reviewer node — Strict Editor persona.
    
    - Reads:  'draft', 'topic', 'research_notes', 'revision_count'
    - Writes: 'review_score', 'feedback'
    
    Scoring rubric (each category 0-2 points = 10 points total):
      1. Accuracy      — Does it match the research notes?
      2. Clarity       — Is it easy to understand?
      3. Structure     — Does it flow logically?
      4. Engagement    — Would a reader keep reading?
      5. Completeness  — Does it cover the topic fully?
    """
    print("\n🔎 REVIEWER NODE ACTIVATED")
    
    # Track which revision this is
    revision_count = state.get("revision_count", 0)
    print(f"📋 Reviewing draft (revision #{revision_count})")
    
    draft          = state["draft"]
    topic          = state["topic"]
    research_notes = state.get("research_notes", "")
    
    # ── The Strict Editor System Prompt ───────────────────────────────
    system_prompt = """You are ALEX, a ruthlessly strict senior editor at a top publication.

    Your reputation depends on only publishing exceptional content.
    You do NOT give high scores easily — a 10 is a masterpiece, a 7 means real problems exist.

    Your scoring rubric (each category scored 0-2):
    ┌─────────────────┬────────────────────────────────────────────────┐
    │ Category        │ What you're checking                           │
    ├─────────────────┼────────────────────────────────────────────────┤
    │ Accuracy (0-2)  │ Facts align with research, no hallucinations   │
    │ Clarity (0-2)   │ Plain language, no jargon without explanation  │
    │ Structure (0-2) │ Clear intro, body, conclusion — logical flow   │
    │ Engagement (0-2)│ Hook, storytelling, would reader continue?     │
    │ Completeness(0-2)│ Covers topic fully, no obvious gaps           │
    └─────────────────┴────────────────────────────────────────────────┘
    Total score = sum of all categories (max 10)

    You MUST respond in valid JSON. No text before or after the JSON block."""

    # ── The Review Prompt ─────────────────────────────────────────────
    review_prompt = f"""Review this article draft.

    ASSIGNED TOPIC: {topic}

    RESEARCH NOTES THAT SHOULD BE USED:
    {research_notes}

    DRAFT TO REVIEW:
    {draft}

    Respond ONLY with this exact JSON structure:
    {{
        "scores": {{
            "accuracy": <0-2>,
            "clarity": <0-2>,
            "structure": <0-2>,
            "engagement": <0-2>,
            "completeness": <0-2>
        }},
        "total_score": <sum of all scores, 0-10>,
        "strengths": ["<specific strength 1>", "<specific strength 2>"],
        "critical_issues": ["<specific issue 1>", "<specific issue 2>"],
        "feedback": "<detailed, actionable feedback telling the writer EXACTLY what to fix>"
    }}"""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=review_prompt),
    ]
    
    print("🤖 Editor Alex reviewing the draft...")
    response = llm.invoke(messages)
    raw_response = response.content
    
    # ── Robust JSON Parsing ───────────────────────────────────────────
    # LLMs sometimes add markdown fences around JSON — handle that
    review_data = _parse_review_json(raw_response)
    
    score    = review_data["total_score"]
    feedback = review_data["feedback"]
    
    # ── Display the Review Report ─────────────────────────────────────
    _display_review_report(review_data, revision_count)
    
    return {
        "review_score": score,
        "feedback":     feedback,
        "review_data":  review_data,   # Store full breakdown in state
    }


def _parse_review_json(raw_response: str) -> dict:
    """
    Robustly parse the JSON response from the reviewer LLM.
    Handles markdown code fences, extra whitespace, etc.
    """
    # Strategy 1: Direct JSON parse
    try:
        return json.loads(raw_response.strip())
    except json.JSONDecodeError:
        pass
    
    # Strategy 2: Extract JSON from markdown code fences
    # Matches ```json ... ``` or ``` ... ```
    json_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw_response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass
    
    # Strategy 3: Find anything that looks like a JSON object
    json_match = re.search(r"\{.*\}", raw_response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    
    # Strategy 4: Full fallback — parse score from text manually
    print("⚠️  JSON parsing failed — using fallback parser")
    score = 5  # Default middle score
    
    # Try to find a number after "score" or "total"
    score_match = re.search(r'(?:total_score|score)["\s:]+(\d+)', raw_response, re.IGNORECASE)
    if score_match:
        score = int(score_match.group(1))
    
    return {
        "scores": {"accuracy": 1, "clarity": 1, "structure": 1, "engagement": 1, "completeness": 1},
        "total_score": score,
        "strengths": ["Could not parse strengths"],
        "critical_issues": ["Could not parse issues"],
        "feedback": raw_response  # Use the full raw response as feedback
    }


def _display_review_report(review_data: dict, revision_count: int):
    """Pretty-print the review report to the console."""
    
    score   = review_data["total_score"]
    scores  = review_data.get("scores", {})
    
    # Score bar visualization
    filled = "█" * score
    empty  = "░" * (10 - score)
    bar    = f"{filled}{empty}"
    
    # Verdict based on score
    if score >= 8:
        verdict = "✅ APPROVED"
    elif score >= 6:
        verdict = "⚠️  NEEDS REVISION"
    else:
        verdict = "❌ SIGNIFICANT REVISION REQUIRED"
    
    print(f"\n{'─'*50}")
    print(f"  📊 EDITORIAL REVIEW REPORT (Revision #{revision_count})")
    print(f"{'─'*50}")
    print(f"  Overall Score: [{bar}] {score}/10")
    print(f"  Verdict:       {verdict}")
    print(f"{'─'*50}")
    
    if scores:
        print("  Category Breakdown:")
        categories = {
            "accuracy":     "Accuracy    ",
            "clarity":      "Clarity     ",
            "structure":    "Structure   ",
            "engagement":   "Engagement  ",
            "completeness": "Completeness",
        }
        for key, label in categories.items():
            cat_score = scores.get(key, "?")
            cat_bar   = "●" * int(cat_score) + "○" * (2 - int(cat_score))
            print(f"    {label}: [{cat_bar}] {cat_score}/2")
    
    print(f"\n  ✨ Strengths:")
    for s in review_data.get("strengths", []):
        print(f"    + {s}")
    
    print(f"\n  🚨 Critical Issues:")
    for issue in review_data.get("critical_issues", []):
        print(f"    - {issue}")
    
    print(f"\n  📝 Editor's Feedback:")
    print(f"    {review_data.get('feedback', 'No feedback')[:200]}...")
    print(f"{'─'*50}")

### Publisher node

In [ ]:
# ============================================================
# 🆕 PUBLISHER NODE — The final step (guarded by human approval)
# ============================================================
#
# This is intentionally simple — it represents the "publish" action.
# In a real system this could:
#   → Post to a CMS (WordPress, Ghost, etc.)
#   → Push to a database
#   → Send via API to social media
#   → Create a PDF and email it
#
# For now, it just prints a confirmation.
# The KEY point is: the graph will PAUSE before this node
# and wait for human approval.
# ============================================================

def publisher_node(state: ContentState) -> dict:
    """
    Publisher node — the final action in the pipeline.
    
    This node only executes AFTER a human has approved.
    The graph pauses (interrupts) before reaching this node.
    """
    print("\n" + "🟢" * 30)
    print("  📰 PUBLISHED!")
    print("  " + "─" * 50)
    print(f"  Title Topic: {state['topic']}")
    print(f"  Final Score:  {state['review_score']}/10")
    print(f"  Revisions:    {state['revision_count'] - 1}")
    print(f"  Draft Length:  {len(state['draft'])} characters")
    print("  " + "─" * 50)
    print(f"\n{state['draft']}")
    print("\n  " + "─" * 50)
    print("  ✅ Content is now LIVE!")
    print("🟢" * 30 + "\n")

    return {}  # No state changes needed — just a side effect

### Conditional edge logic

In [ ]:
# ============================================================
# ROUTING FUNCTION (same from Step 3)
# ============================================================

APPROVAL_THRESHOLD = 8
MAX_REVISION_LIMIT = 3

def route_after_review(state: ContentState) -> str:
    score = state.get("review_score", 0)
    revision_count = state.get("revision_count", 0)

    print(f"\n🔀 ROUTING: Score {score}/10 | Revision {revision_count}/{MAX_REVISION_LIMIT}")

    if revision_count >= MAX_REVISION_LIMIT:
        print("  ⏹️  Max revisions — moving to publish")
        return "approved"
    if score >= APPROVAL_THRESHOLD:
        print("  ✅ Approved — moving to publish")
        return "approved"

    print(f"  🔄 Revise — {APPROVAL_THRESHOLD - score} points short")
    return "revise"

### Building blocks

In [ ]:
# ============================================================
# BUILD THE GRAPH WITH SQLITE CHECKPOINTER + INTERRUPT
# ============================================================
# This is where the magic happens.
#
# Two new things in .compile():
#   checkpointer       → WHERE to save state (SQLite file)
#   interrupt_before    → WHICH node to pause before
# ============================================================

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# ── Create the SQLite checkpointer ───────────────────────
# This creates (or opens) a local SQLite database file
# Every state checkpoint gets saved here automatically
sqlite_conn = sqlite3.connect("content_engine_checkpoints.db", check_same_thread=False)
checkpointer = SqliteSaver(sqlite_conn)

# ── Build the graph ──────────────────────────────────────
graph_builder = StateGraph(ContentState)

# Register nodes
graph_builder.add_node("researcher", researcher_node)
graph_builder.add_node("writer",     writer_node)
graph_builder.add_node("reviewer",   reviewer_node)
graph_builder.add_node("publisher",  publisher_node)   # ← NEW

# Linear edges
graph_builder.add_edge(START,        "researcher")
graph_builder.add_edge("researcher", "writer")
graph_builder.add_edge("writer",     "reviewer")

# ── Conditional edge — UPDATED routing ─────────────────────
# Now routes to "publisher" instead of END when approved
graph_builder.add_conditional_edges(
    "reviewer",
    route_after_review,
    {
        "revise":   "writer",
        "approved": "publisher",   # ← Changed from END to publisher
    }
)

# Publisher → END
graph_builder.add_edge("publisher", END)

# ── Compile with checkpointer and interrupt ───────────────
#
# interrupt_before=["publisher"] means:
#   "After the reviewer approves, STOP before running publisher.
#    Save the full state to SQLite. Wait for human to resume."
#
graph = graph_builder.compile(
    checkpointer     = checkpointer,
    interrupt_before = ["publisher"],   # ← PAUSE before publishing
)

print("✅ Graph compiled with:")
print("   📁 Checkpointer: SQLite (content_engine_checkpoints.db)")
print("   ⏸️  Interrupt:    Before 'publisher' node")

### Visualize the graph

In [ ]:
# LangGraph can render your graph as an image — super useful for debugging!

from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # If the image rendering fails, print the text version
    print("Graph structure (text):")
    graph.get_graph().print_ascii()
    print(f"\n(Image rendering requires additional deps: {e})")

### Run

In [ ]:
# ============================================================
# PHASE 1: Run the pipeline — it will PAUSE before publishing
# ============================================================

import uuid

# Generate a unique thread ID for this run
thread_id = f"article-{uuid.uuid4().hex[:8]}"

# The config that links this execution to a checkpointed thread
config = {"configurable": {"thread_id": thread_id}}

print(f"🔑 Thread ID: {thread_id}")
print(f"   (save this — you'll need it to resume!)\n")

initial_state = {
    "topic":            "what is the future of AI?",
    "research_notes":   "",
    "draft":            "",
    "review_score":     None,
    "feedback":         "",
    "review_data":      {},
    "revision_count":   0,
    "revision_history": [],
}

print("🚀 STARTING CONTENT ENGINE (will pause before publishing)")
print("═" * 65)

# .invoke() will run all nodes UNTIL it hits the interrupt
# Then it saves state to SQLite and returns
result = graph.invoke(initial_state, config=config)

print("\n" + "═" * 65)
print("⏸️  PIPELINE PAUSED — Waiting for human approval")
print("═" * 65)

In [ ]:
# ============================================================
# INSPECT THE PAUSED STATE
# ============================================================
# The graph saved its state to SQLite. Let's peek at it.
#
# graph.get_state(config) returns a StateSnapshot object with:
#   .values    → the current state dict
#   .next      → tuple of node names that will run next (when resumed)
#   .config    → the config used
#   .metadata  → execution metadata
# ============================================================

snapshot = graph.get_state(config)

print("📸 CHECKPOINT SNAPSHOT")
print("─" * 50)
print(f"Thread ID:     {thread_id}")
print(f"Next node(s):  {snapshot.next}")       # Should show ('publisher',)
print(f"Topic:         {snapshot.values['topic']}")
print(f"Review Score:  {snapshot.values['review_score']}/10")
print(f"Revisions:     {snapshot.values['revision_count'] - 1}")
print(f"Draft length:  {len(snapshot.values['draft'])} chars")

print(f"\n{'─'*50}")
print("📝 DRAFT FOR HUMAN REVIEW:")
print(f"{'─'*50}")
print(snapshot.values['draft'])

print(f"\n{'─'*50}")
print(f"⭐ Reviewer's Score:    {snapshot.values['review_score']}/10")
print(f"💬 Reviewer's Feedback: {snapshot.values['feedback'][:200]}")
print(f"{'─'*50}")

In [ ]:
# ============================================================
# HUMAN APPROVAL GATE
# ============================================================
# This is where the human decides:
#   "approve"  → Resume the graph, publisher runs
#   "reject"   → Don't resume (or you could update state and re-run)
#
# In a real app, this could be:
#   → A web form with Approve/Reject buttons
#   → A Slack message with reaction emojis
#   → An email with approve/reject links
#   → A CLI prompt (what we're doing here)
# ============================================================

print("╔══════════════════════════════════════════════════════╗")
print("║           🧑 HUMAN APPROVAL REQUIRED                ║")
print("╠══════════════════════════════════════════════════════╣")
print("║                                                      ║")
print("║  The AI pipeline has completed its work.             ║")
print("║  Review the draft above and decide:                  ║")
print("║                                                      ║")
print("║    Type 'approve'  → Publish the content             ║")
print("║    Type 'reject'   → Discard (do not publish)        ║")
print("║                                                      ║")
print("╚══════════════════════════════════════════════════════╝")

# decision = input("\n👉 Your decision: ").strip().lower()

decision = "approve"

if decision == "approve":
    print("\n✅ Human approved! Resuming the pipeline...\n")

    # ── RESUME: invoke with None + same config ─────────────
    # Passing None as input means "don't add new input,
    # just continue from where you paused"
    # The config with the SAME thread_id tells LangGraph
    # which checkpoint to load
    final_state = graph.invoke(None, config=config)

    print("\n🎉 PIPELINE COMPLETE — Content has been published!")

elif decision == "reject":
    print("\n❌ Human rejected. Content was NOT published.")
    print("   The checkpoint is still saved — you can review again later.")
    print(f"   Thread ID: {thread_id}")
    final_state = snapshot.values  # Keep the state for inspection

else:
    print(f"\n⚠️  Unknown input: '{decision}'")
    print("   Run this cell again and type 'approve' or 'reject'")
    final_state = snapshot.values